In [2]:
import os
import pandas as pd
import lyricsgenius
import time
import random
from dotenv import load_dotenv


# -----------------------------
# CONFIGURATION
# -----------------------------

# Load environment variables from .env file (using absolute path for reliability)
env_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))), '.env')
load_dotenv(env_path, override=True, encoding='utf-8')

# 1) Genius API Token
GENIUS_API_TOKEN = os.getenv("GENIUS_API_TOKEN")
if not GENIUS_API_TOKEN:
    raise ValueError("GENIUS_API_TOKEN not found in .env file")

# 2) Path to your artist list (one artist name per line)
ARTIST_LIST_PATH = "another_artists_list_300.txt"

# 3) Output CSV (where we'll append results as we go)
OUTPUT_CSV = "scraped_lyrics_2.csv"


# 4) How many songs to fetch per artist
SONGS_PER_ARTIST = int(os.getenv("SONGS_PER_ARTIST", "25"))
print(f"Will fetch up to {SONGS_PER_ARTIST} songs per artist")

# 5) Pause (seconds) between artist requests to avoid rate-limiting
SLEEP_BETWEEN_ARTISTS = float(os.getenv("SLEEP_BETWEEN_ARTISTS", "1.5"))
print(f"Will sleep {SLEEP_BETWEEN_ARTISTS} seconds between artist requests")

# 6) Rate limit handling configuration
INITIAL_BACKOFF = int(os.getenv("INITIAL_BACKOFF", 10))  # Start with 10 seconds
MAX_RETRIES = int(os.getenv("MAX_RETRIES", 5))       # Try up to 5 times

print(f"Using initial backoff of {INITIAL_BACKOFF}s with {MAX_RETRIES} max retries")
# -----------------------------
# INITIALIZE GENIUS CLIENT
# -----------------------------

# Initialize lyricsgenius.Genius with some options
genius = lyricsgenius.Genius(
    GENIUS_API_TOKEN,
    timeout=15,
    retries=3,
    sleep_time=0.25,  # small pause between each page scrape
    excluded_terms=["(Remix)", "(Live)"],  # Exclude these terms from song titles
    skip_non_songs=True,  # Skip non-song entries (e.g., interviews)
  # Remove section headers like "Verse", "Chorus"
)


# -----------------------------
# RATE LIMIT HANDLER
# -----------------------------
def with_rate_limit_handling(api_function):
    """Decorator to handle rate limit errors with exponential backoff"""
    def wrapper(*args, **kwargs):
        for attempt in range(MAX_RETRIES + 1):
            try:
                return api_function(*args, **kwargs)
            except Exception as e:
                error_str = str(e)
                # Check if it's a rate limit error
                if "429" in error_str and attempt < MAX_RETRIES:
                    # Calculate backoff time with jitter
                    backoff_time = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(1, 5)
                    print(f"\nRate limit exceeded. Waiting {backoff_time:.1f} seconds before retry {attempt+1}/{MAX_RETRIES}")
                    time.sleep(backoff_time)
                else:
                    if "429" in error_str:
                        print(f"\nRate limit exceeded after {MAX_RETRIES} retries. Consider increasing wait time.")
                    raise
    return wrapper



# -----------------------------
# HELPER FUNCTION: fetch_artist_lyrics
# -----------------------------
@with_rate_limit_handling
def search_artist(artist_name, max_songs):
    """Search for an artist with rate limit handling"""
    return genius.search_artist(artist_name, max_songs=max_songs, sort="popularity",get_full_info=False)

@with_rate_limit_handling
def search_song(title, artist):
    """Search for a song with rate limit handling"""
    return genius.search_song(title=title, artist=artist, get_full_info=False)


# Add this function after your imports and before the GENIUS CLIENT section


def fetch_artist_lyrics(artist_name, max_songs=SONGS_PER_ARTIST):
    """
    Fetch up to max_songs tracks for `artist_name`, returning a list of dicts
    """
    songs_data = []
    try:
        # Search for the artist with rate limit handling
        artist_obj = search_artist(artist_name, max_songs)
        
        if artist_obj is None or not artist_obj.songs:
            print(f"  → No songs found for artist: {artist_name}")
            return songs_data

        for song in artist_obj.songs:
            title = song.title.strip()
            lyrics = song.lyrics.strip()
            
            # Skip extremely short lyrics (e.g., < 20 chars)
            if len(lyrics) < 20:
                continue
            songs_data.append({
                "artist": artist_name,
                "song_title": title,
                "lyrics": lyrics
            })
            
    except Exception as e:
        print(f"ERROR: Could not search for artist [{artist_name}]: {e}")
        
    return songs_data

def main():
    # 1) Read existing CSV (if any), so we don't re‐scrape duplicates
    if os.path.exists(OUTPUT_CSV):
        master_df = pd.read_csv(OUTPUT_CSV, encoding='utf-8')
        # master_df = safe_read_csv(OUTPUT_CSV)
        # Create a set of (artist, song_title) for quick "already scraped" checks
        existing_pairs = set(zip(master_df["artist"], master_df["song_title"]))
        
        # Check which artists have already met their quota
        artist_song_counts = master_df.groupby('artist').size()
        complete_artists = set(artist_song_counts[artist_song_counts >= SONGS_PER_ARTIST].index)
        incomplete_artists = set(artist_song_counts[artist_song_counts < SONGS_PER_ARTIST].index)
        
        print(f"Loaded {len(master_df)} existing rows from {OUTPUT_CSV}")
        print(f"Complete artists (>= {SONGS_PER_ARTIST} songs): {len(complete_artists)}")
        print(f"Incomplete artists (< {SONGS_PER_ARTIST} songs): {len(incomplete_artists)}")
    else:
        master_df = pd.DataFrame(columns=["artist", "song_title", "lyrics"])
        existing_pairs = set()
        complete_artists = set()
        incomplete_artists = set()
        print(f"No existing CSV found. A new one will be created: {OUTPUT_CSV}")

    # 2) Read artist list
    with open(ARTIST_LIST_PATH, "r", encoding="utf-8") as f:
        artists = [line.strip() for line in f if line.strip()]
    print(f"Read {len(artists)} artists from {ARTIST_LIST_PATH}")

    # 3) Filter artists: skip complete ones, include incomplete and new ones
    artists_to_scrape = [artist for artist in artists if artist not in complete_artists]
    skipped_count = len(artists) - len(artists_to_scrape)
    
    print(f"Will scrape {len(artists_to_scrape)} artists (skipping {skipped_count} completed artists)")
    if incomplete_artists:
        print(f"Resuming scraping for {len(incomplete_artists)} incomplete artists")

    # 4) Loop over each artist that needs scraping
    for idx, artist_name in enumerate(artists_to_scrape, 1):
        # Check if this is a resume case
        if artist_name in incomplete_artists:
            current_count = len([pair for pair in existing_pairs if pair[0] == artist_name])
            remaining_needed = SONGS_PER_ARTIST - current_count
            print(f"[{idx}/{len(artists_to_scrape)}] Resuming artist: {artist_name} (has {current_count}, needs {remaining_needed} more) ", end="")
        else:
            print(f"[{idx}/{len(artists_to_scrape)}] Scraping new artist: {artist_name} ", end="")
        
        fetched = fetch_artist_lyrics(artist_name, max_songs=SONGS_PER_ARTIST)

        # Filter out any (artist, song) pairs we already have
        new_rows = []
        for item in fetched:
            key = (item["artist"], item["song_title"])
            if key in existing_pairs:
                continue
            new_rows.append(item)
            existing_pairs.add(key)

        # 5) Append new_rows to master_df (and save immediately)
        if new_rows:
            new_df = pd.DataFrame(new_rows)
            master_df = pd.concat([master_df, new_df], ignore_index=True)

            # Sort by artist for better organization
            master_df = master_df.sort_values(['artist', 'song_title']).reset_index(drop=True)

            # Save after each artist to avoid data loss if script crashes
            master_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
            print(f"→ Retrieved {len(new_rows)} new songs (total now {len(master_df)})")
        else:
            print("→ No new songs found or all songs already exist.")

        # 6) Sleep to avoid hitting rate limits
        time.sleep(SLEEP_BETWEEN_ARTISTS)

    # Final sorting and statistics
    master_df = master_df.sort_values(['artist', 'song_title']).reset_index(drop=True)
    master_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

    print("\nScraping complete.")
    print(f"Final row count: {len(master_df)}")
    print(f"Distinct artists in CSV: {master_df['artist'].nunique()}")
    print(f"Distinct songs in CSV: {master_df['song_title'].nunique()}")
    
    # Show final artist statistics
    final_artist_counts = master_df.groupby('artist').size().sort_values(ascending=False)
    print(f"\nTop 10 artists by song count:")
    print(final_artist_counts.head(10))
    
    # Show artists that still need more songs
    incomplete_final = final_artist_counts[final_artist_counts < SONGS_PER_ARTIST]
    if len(incomplete_final) > 0:
        print(f"\nArtists still needing more songs ({len(incomplete_final)} total):")
        print(incomplete_final.head(10))

if __name__ == "__main__":
    main()

Will fetch up to 10 songs per artist
Will sleep 0.25 seconds between artist requests
Using initial backoff of 5s with 5 max retries
No existing CSV found. A new one will be created: scraped_lyrics_2.csv
Read 300 artists from another_artists_list_300.txt
Will scrape 300 artists (skipping 0 completed artists)
[1/300] Scraping new artist: Louis Armstrong Searching for songs by Louis Armstrong...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "La Vie En Rose"
Song 2: "What a Wonderful World"
Song 3: "Go Down Moses"
Song 4: "Jeepers Creepers"
Song 5: "Hello, Dolly!"
Song 6: "A Kiss To Build a Dream On"
Song 7: "Mack the Knife"
Song 8: "We Have All the Time in the World"
Song 9: "When the Saints Go Marching In"
Song 10: "It Don’t Mean a Thing (If It Ain’t Got That Swing)"

Reached user-specified song limit (10).
Done. Found 10 songs.
→ Retrieved 10 new songs (total now 10)
[2/300] Scraping new artist: Duke Ellington Searching for songs by Duke Ellington...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "Don’t Get Around Much Anymore"
Song 2: "Satin Doll"
Song 3: "Sophisticated Lady"
Song 4: "Mood Indigo"
Song 5: "Do Nothing Till You Hear from Me"
Song 6: "It Don’t Mean a Thing (If it Ain’t Got That Swing)"
Song 7: "Lambeth Walk"
Song 8: "Jump for Joy"
Song 9: "Come Sunday"
Song 10: "Just Squeeze Me (But Don’t Tease Me)"

Reached user-specified song limit (10).
Done. Found 10 songs.
→ Retrieved 10 new songs (total now 20)
[3/300] Scraping new artist: John Coltrane Searching for songs by John Coltrane...



...

Song 1: "Hungry Hippo"
Song 2: "Meagan Good"
Song 3: "Unemployed"
Song 4: "Pretty Ugly"
Song 5: "Only Child"
Song 6: "MUMBO JUMBO"
Song 7: "Flea Market"
Song 8: "Body Of Water"
Song 9: "Fruit Salad"
Song 10: "Hookers"

Reached user-specified song limit (10).
Done. Found 10 songs.
→ Retrieved 10 new songs (total now 2951)
[299/300] Scraping new artist: Rico Nasty Searching for songs by Rico Nasty...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "Smack a Bitch"
"#PROUDCATOWNERREMIX" is not valid. Skipping.
"ringtone (Remix)" is not valid. Skipping.
Song 2: "Poppin"
"Iggady (Remix)" is not valid. Skipping.
Song 3: "Beat My Face"
Song 4: "Countin’ Up"
Song 5: "OHFR?"
Song 6: "Key Lime OG"
Song 7: "Trust Issues"
Song 8: "Big Titties"
Song 9: "Hey Arnold"
"Smack A Bitch (Remix)" is not valid. Skipping.
Song 10: "Rage"

Reached user-specified song limit (10).
Done. Found 10 songs.
→ Retrieved 10 new songs (total now 2961)
[300/300] Scraping new artist: Doechii Searching for songs by Doechii...



d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\lyricsgenius\genius.py:589: FutureWarning: The constructor signature will change in a future version. It will change to Song(lyrics, body) instead of Song(client, json_dict, lyrics).
  song = Song(self, song_info, lyrics)


Song 1: "Anxiety"
Song 2: "DENIAL IS A RIVER"
Song 3: "What It Is (Solo Version)"
Song 4: "NISSAN ALTIMA"
Song 5: "What It Is (Block Boy)"
Song 6: "CATFISH"
Song 7: "Yucky Blucky Fruitcake"
Song 8: "Nosebleeds"
Song 9: "Alter Ego"
Song 10: "BOILED PEANUTS"

Reached user-specified song limit (10).
Done. Found 10 songs.
→ Retrieved 10 new songs (total now 2971)

Scraping complete.
Final row count: 2971
Distinct artists in CSV: 298
Distinct songs in CSV: 2910

Top 10 artists by song count:
artist
50 Cent                              10
Peter Tosh                           10
Pet Shop Boys                        10
Perfume Genius                       10
Patsy Cline                          10
PSY                                  10
PJ Harvey                            10
Outkast                              10
Orchestral Manoeuvres in the Dark    10
Orbital                              10
dtype: int64

Artists still needing more songs (2 total):
artist
Ali Farka Touré    6
The Ramones    

In [3]:
# Clean lyrics by removing metadata and save to new CSV
import pandas as pd
import re
import os


def clean_lyrics_metadata(lyrics_text):
    """
    Clean lyrics by finding the first structural marker [Something] or "Read More" and keeping everything from there.

    Args:
        lyrics_text (str): Raw lyrics text with metadata

    Returns:
        str: Cleaned lyrics starting from first structural marker or after "Read More"
    """
    if pd.isna(lyrics_text) or not lyrics_text:
        return ""

    text = str(lyrics_text)

    # Pattern 1: Find "Read More" and take everything after it
    read_more_pattern = r"read more\s*"
    read_more_match = re.search(read_more_pattern, text, re.IGNORECASE)

    # Pattern 2: Find structural markers [Something]
    structural_pattern = r"\[(Intro|Chorus|Verse|Pre-Chorus|Bridge|.*).*?\]"
    structural_match = re.search(structural_pattern, text)

    # Pattern 3: Find "Lyrics" followed by any alphabetic character
    lyrics_pattern = r"Lyrics(?=[\S])"
    lyrics_match = re.search(lyrics_pattern, text)

   
    read_more_pos = read_more_match.start() if read_more_match else None
    structural_pos = structural_match.start() if structural_match else None
    lyrics_pos = lyrics_match.start() if lyrics_match else None

    if read_more_pos:
        # If "Read More" appears => just take everything after it
        start_index = read_more_match.end()
        cleaned_text = text[start_index:].strip()
        return cleaned_text
    elif structural_match:
        # Structural marker appears first - take everything from it
        start_index = structural_match.start()
        cleaned_text = text[start_index:].strip()
        return cleaned_text
    elif lyrics_pos:
        # "Lyrics" appears first - take everything from it
        start_index = lyrics_match.end()
        cleaned_text = text[start_index:].strip()
        return cleaned_text
    else:
        # Neither pattern found, return original text
        return text.strip()


# Load the scraped lyrics
print("📁 Loading scraped lyrics data...")
input_csv = OUTPUT_CSV

if not os.path.exists(input_csv):
    print(f"❌ Error: {input_csv} not found!")
    print("💡 Make sure you've run the scraping process first.")
else:
    df = pd.read_csv(input_csv, encoding='utf-8')
    print(f"✅ Loaded {len(df)} songs from {input_csv}")
    
    # Apply cleaning function
    print(f"\n🧹 Cleaning lyrics metadata...")
    df['lyrics_cleaned'] = df['lyrics'].apply(clean_lyrics_metadata)
    
    # Show results after cleaning
    print(f"\n📝 Sample lyrics AFTER cleaning:")
    for i in range(min(3, len(df))):
        original_length = len(str(df.iloc[i]['lyrics']))
        cleaned_length = len(str(df.iloc[i]['lyrics_cleaned']))
        chars_removed = original_length - cleaned_length
        
        print(f"\n🎵 {df.iloc[i]['artist']} - {df.iloc[i]['song_title']}")
        print(f"   Cleaned (first 150 chars): {df.iloc[i]['lyrics_cleaned'][:150]}...")
        print(f"   Length: {original_length} → {cleaned_length} chars (removed {chars_removed})")
    
    # Calculate cleaning statistics
    original_lengths = [len(str(lyrics)) for lyrics in df['lyrics']]
    cleaned_lengths = [len(str(lyrics)) for lyrics in df['lyrics_cleaned']]
    
    total_chars_removed = sum(original_lengths) - sum(cleaned_lengths)
    avg_chars_removed = total_chars_removed / len(df)
    
    print(f"\n📊 Cleaning Statistics:")
    print(f"   Total characters removed: {total_chars_removed:,}")
    print(f"   Average characters removed per song: {avg_chars_removed:.1f}")
    print(f"   Percentage of text removed: {(total_chars_removed / sum(original_lengths)) * 100:.1f}%")
    
    # Check for songs where no cleaning occurred (no structural markers)
    no_change_count = sum(1 for i in range(len(df)) if df.iloc[i]['lyrics'] == df.iloc[i]['lyrics_cleaned'])
    print(f"   Songs with no structural markers: {no_change_count} ({no_change_count/len(df)*100:.1f}%)")
    
    
    base_name = os.path.splitext(OUTPUT_CSV)[0]  # "scraped_lyrics_2"
    output_csv = f"{base_name}_no_metadata.csv"
    # Save to new CSV"
    
    
    # Create new DataFrame with cleaned lyrics
    cleaned_df = df[['artist', 'song_title', 'lyrics_cleaned']].copy()
    cleaned_df = cleaned_df.rename(columns={'lyrics_cleaned': 'lyrics'})
    
    # Save cleaned data
    cleaned_df.to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"\n💾 Saved cleaned lyrics to: {output_csv}")
    print(f"✅ Processing complete!")
    
    # Show final dataset info
    print(f"\n📋 Final Dataset Info:")
    print(f"   File: {output_csv}")
    print(f"   Songs: {len(cleaned_df):,}")
    print(f"   Artists: {cleaned_df['artist'].nunique():,}")
    print(f"   Columns: {list(cleaned_df.columns)}")
    
    # Verify the cleaning worked by showing first few characters of cleaned lyrics
    print(f"\n🔍 Verification - First characters of cleaned lyrics:")
    for i in range(min(5, len(cleaned_df))):
        lyrics_start = cleaned_df.iloc[i]['lyrics'][:50]
        print(f"   {i+1}. {lyrics_start}...")
        
    print(f"\n🎉 Metadata removal complete! Use '{output_csv}' for your topic modeling.")

📁 Loading scraped lyrics data...
✅ Loaded 2971 songs from scraped_lyrics_2.csv

🧹 Cleaning lyrics metadata...

📝 Sample lyrics AFTER cleaning:

🎵 50 Cent - 21 Questions
   Cleaned (first 150 chars): [Intro: 50 Cent]
New York City
You are now rockin'
With 50 Cent
You gotta love it

[Verse 1: 50 Cent]
I just wanna chill and twist the lye
Catch stunt...
   Length: 3399 → 3084 chars (removed 315)

🎵 50 Cent - Best Friend
   Cleaned (first 150 chars): [Intro]
Yeah!
It's my tape, man
Listen to my tape
(I've waited
I've waited
Time went by
But all I could do is cry
Silly, silly) Woo!

[Chorus]
If I wa...
   Length: 4238 → 3986 chars (removed 252)

🎵 50 Cent - Candy Shop
   Cleaned (first 150 chars): [Intro: 50 Cent]
Yeah, uh-huh
So seductive

[Chorus: 50 Cent & Olivia]
I'll take you to the candy shop
I'll let you lick the lollipop
Go 'head, girl, ...
   Length: 3313 → 3015 chars (removed 298)

📊 Cleaning Statistics:
   Total characters removed: 493,432
   Average characters removed per song: 